# Day 6 Capstone Mini-Challenge (Instructor Solution) — Complete Config-Driven ML Pipeline

Complete end-to-end reference solution connecting all Day 6 modules:
`YAML Config` → `Ingestion` → `Validation` → `Feature Engineering` → `Train/Test Pipeline` → `Artifact Serialization` → `Experiment Tracking`.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
import uuid

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

with open("configs/train_config.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("Loaded project:", CFG["project"]["name"])


In [ ]:
def stage_ingest(cfg: dict) -> tuple[pd.DataFrame, str]:
    cust_path = Path(cfg["data"]["customers_csv"])
    usage_path = Path(cfg["data"]["usage_json"])

    data_hash = hashlib.sha256(cust_path.read_bytes()).hexdigest()[:12]

    cust = pd.read_csv(cust_path)
    with open(usage_path, "r", encoding="utf-8") as f:
        usage = pd.DataFrame(json.load(f))

    cust["plan_tier"] = cust["plan_tier"].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
    cust["country"] = cust["country"].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
    cust = cust.drop_duplicates(subset=[cfg["data"]["id_column"]], keep="first")

    merged = cust.merge(usage, on=cfg["data"]["id_column"], how="left", validate="one_to_one")
    return merged, data_hash


def stage_validate(df: pd.DataFrame, cfg: dict) -> dict:
    rules = cfg["validation"]
    errors = []

    if not df[cfg["data"]["id_column"]].is_unique:
        errors.append("Duplicate primary keys found.")
    if not df["age"].dropna().between(rules["min_age"], rules["max_age"]).all():
        errors.append("Age values outside configured bounds.")
    if (df["tenure_months"].dropna() < rules["min_tenure_months"]).any():
        errors.append("Negative tenure_months found.")
    if not set(df["plan_tier"].dropna().unique()).issubset(set(rules["allowed_plan_tiers"])):
        errors.append("Unexpected plan_tier values found.")

    if errors:
        raise ValueError("Data validation failed: " + "; ".join(errors))

    return {"passed": True, "rows": int(len(df))}


def stage_engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["charges_per_login"] = out["monthly_charges"] / (out["login_days_30d"] + 1.0)
    out["tickets_per_tenure_month"] = out["support_tickets_30d"] / (out["tenure_months"] + 1.0)
    out["watch_hours_per_login"] = out["watch_hours_30d"] / (out["login_days_30d"] + 1.0)
    out["is_new_customer"] = (out["tenure_months"] <= 3).astype(int)
    return out


def _make_model(model_type: str, params: dict, seed: int):
    if model_type == "logistic_regression":
        return LogisticRegression(random_state=seed, **params)
    if model_type == "random_forest":
        return RandomForestClassifier(random_state=seed, **params)
    if model_type == "gradient_boosting":
        return GradientBoostingClassifier(random_state=seed, **params)
    raise ValueError(f"Unsupported model_type: {model_type}")


def stage_train_and_log(df: pd.DataFrame, data_hash: str, cfg: dict, log_path: Path) -> dict:
    seed = cfg["reproducibility"]["random_seed"]
    random.seed(seed)
    np.random.seed(seed)

    num_cols = cfg["features"]["numeric_columns"]
    cat_cols = cfg["features"]["categorical_columns"]
    target = cfg["data"]["target_column"]

    X = df[num_cols + cat_cols]
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg["reproducibility"]["test_size"], random_state=seed, stratify=y
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                cat_cols,
            ),
        ]
    )
    model = _make_model(cfg["model"]["type"], cfg["model"]["params"], seed)
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])

    cv = StratifiedKFold(n_splits=cfg["reproducibility"]["cv_folds"], shuffle=True, random_state=seed)
    cv_auc = float(cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc").mean())

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    run_id = uuid.uuid4().hex[:8]
    artifact_path = Path(cfg["artifacts"]["output_dir"]) / f"capstone_{run_id}.joblib"
    artifact_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipe, artifact_path)

    record = {
        "run_id": run_id,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_type": cfg["model"]["type"],
        "params": cfg["model"]["params"],
        "random_seed": seed,
        "data_hash": data_hash,
        "cv_roc_auc": round(cv_auc, 4),
        "test_roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
        "test_f1": round(float(f1_score(y_test, y_pred)), 4),
        "artifact_path": str(artifact_path),
    }

    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

    return record


In [ ]:
capstone_log = Path("artifacts/capstone_experiments.jsonl")

raw_merged, data_hash = stage_ingest(CFG)
val_report = stage_validate(raw_merged, CFG)
featured_df = stage_engineer_features(raw_merged)
run_record = stage_train_and_log(featured_df, data_hash, CFG, capstone_log)

print("Capstone Run Record:")
print(json.dumps(run_record, indent=2))

assert val_report["passed"] is True
assert len(data_hash) == 12
assert run_record["cv_roc_auc"] >= 0.72
assert run_record["test_roc_auc"] >= 0.72
assert Path(run_record["artifact_path"]).exists()
assert capstone_log.exists()

reloaded = joblib.load(run_record["artifact_path"])
sample_X = featured_df[CFG["features"]["numeric_columns"] + CFG["features"]["categorical_columns"]].head(3)
sample_probs = reloaded.predict_proba(sample_X)[:, 1]
assert len(sample_probs) == 3 and np.all((sample_probs >= 0) & (sample_probs <= 1))

print("\n[CAPSTONE COMPLETE!] Reloaded sample churn probabilities:", np.round(sample_probs, 3))
